<a href="https://colab.research.google.com/github/Kittichot2003/Ge338/blob/main/Lab_1_6606614631_Pan_sharpening.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Pan-sharpening**

เทคนิค Pan-sharpening เป็นเทคนิคที่ใช้กับภาพถ่ายดาวเทียม ที่มี Panchromatic band เช่น THEOS-1, LANDSAT-8, LANDSAT-9 เพื่อให้มีรายละเอียดและมีประโยชน์มากขึ้น โดย Lab นี้เราจะทดลองกับข้อมูลดาวเทียม Landsat-9

ข้อมูลถ่ายที่เป็น Multi-Spectral มักจะมีรายละเอียดภาพไม่ค่อยดีนัก เมื่อเทียบกับข้อมูล Panchromatics  ดังนั้น การปรับความคมชัดของภาพช่วยได้โดยการรวมภาพเหล่านี้เข้ากับภาพขาวดำพิเศษที่มีรายละเอียดมากขึ้น ภาพขาวดำที่เรียกว่าภาพแพนโครมาติก มีรายละเอียดในระดับที่สูงกว่า แต่ไม่มีสี เราต้องการใช้รายละเอียดจากภาพแพนโครมาติกและสีจากภาพ Landsat-9 เพื่อสร้างภาพสีที่มีรายละเอียดสูงใหม่

ในการดำเนินการใน Lab นี้ นักศึกษา จำเป็นต้องตรวจสอบให้แน่ใจว่าข้อมูลทั้ง Panchromatic และ Multispectral ของ Landsat-9 อยู่ในตำแหน่งที่สมบูรณ์แบบ เพื่อให้เข้ากันได้อย่างลงตัว

มีหลายวิธีในการรวมภาพเหล่านี้ เช่น Brovey Transform, IHS Transform และ PCA วิธีการเหล่านี้ทำให้แน่ใจว่าภาพใหม่จะคงสีจาก Landsat-9 แต่ยังได้รับรายละเอียดเพิ่มเติมจากภาพแพนโครมาติกด้วย


เมื่อเราทำ Pan-sharpening เสร็จแล้ว เราก็จะได้ภาพใหม่ที่เป็นภาพสีที่มีรายละเอียดสูงขึ้น  โดยมีสีทั้งหมดจาก Landsat-9 และความคมชัดพิเศษจากภาพแบบแพนโครมาติก รูปภาพใหม่นี้สามารถนำไปใช้ได้หลายอย่าง เช่น ศึกษาพื้นดิน ค้นหาการเปลี่ยนแปลง หรือเพียงแค่ดูรายละเอียดเพิ่มเติมเกี่ยวกับโลก ดังนั้น การปรับความคมชัดของภาพเป็นวิธีหนึ่งในการทำให้ภาพจากดาวเทียมมีประโยชน์มากขึ้นสำหรับงานสำคัญทุกประเภท

In [3]:
# ทำการ Authenticate และ initialize Earth Engine
import ee
import geemap
ee.Authenticate()
ee.Initialize(project='ee-kittichot6692') #อย่าลืมเปลี่ยนชื่อโปรเจคของตัวเอง

In [4]:
# กำหนดพื้นที่สนใจ
geometry = ee.Geometry.Point([98.95799098999555, 18.84423947416328])

# เรียกภาพ L9 ตัวอย่างแล้วดึง RGB และ Pan ออกมา
image = (ee.ImageCollection("LANDSAT/LC09/C02/T1_TOA")
         .filterDate('2022-01-01', '2022-03-30')
         .filterBounds(geometry)
         .sort('CLOUD_COVER')
         .first())

In [5]:
# ทำการ Pan-Sharp
rgb = image.select('B4', 'B3', 'B2')
pan = image.select('B8')

# แปลงเป็น HSV, สลับในแถบ PAN และแปลงกลับเป็น RGB
huesat = rgb.rgbToHsv().select('hue', 'saturation')
upres = ee.Image.cat([huesat, pan]).hsvToRgb()

In [6]:
# สร้างแผนที่
Map = geemap.Map(center=[18.84423947416328, 98.95799098999555], zoom=14)

# แสดง เลเยอร์ ก่อนและหลังโดยใช้พารามิเตอร์ vis เดียวกัน
Map.addLayer(rgb, {'max': 0.28}, 'Original')
Map.addLayer(pan, {'max': 0.28}, 'Pan')
Map.addLayer(upres, {'max': 0.28}, 'Pansharpened')
Map


Map(center=[18.84423947416328, 98.95799098999555], controls=(WidgetControl(options=['position', 'transparent_b…

คำถาม
ข้อเพื่อทดสอบความเข้าใจของนักศึกษาเกี่ยวกับเทคนิคการทำ Pan-sharpening ด้วยภาพ Landsat-9 จงตอบคำถามต่อไปนี้

1. อะไรคือเป้าหมายหลักของการปรับความคมชัดของภาพในการสำรวจระยะไกล โดยเฉพาะเมื่อใช้ภาพ Landsat-9 อธิบายว่าเหตุใดจึงมีความสำคัญในการประมวลผลภาพ

2. อธิบายขั้นตอนสำคัญที่เกี่ยวข้องกับการปรับความคมชัดด้วนเทคนิค Pan-sharpening ตั้งแต่การรับข้อมูลไปจนถึงการสร้างภาพที่ปรับความคมชัด เทคนิค Pan-sharpening มี กระบวนการสุ่มตัวอย่างใหม่ช่วยจัดแนวภาพหลายสเปกตรัมและภาพแพนโครมาติกอย่างไร


**1.** อะไรคือเป้าหมายหลักของการปรับความคมชัดของภาพในการสำรวจระยะไกล โดยเฉพาะเมื่อใช้ภาพ Landsat-9 อธิบายว่าเหตุใดจึงมีความสำคัญในการประมวลผลภาพ

**Answer**
เป้าหมายหลักของการปรับความคมชัดของภาพในการสำรวจระยะไกลคือการปรับการกระจายค่าระดับความสว่างของพิกเซล ( DN หรือ ค่าการสะท้อน) ให้อยู่ในช่วงที่เหมาะสมกับการแสดงผลและการวิเคราะห์ โดยไม่เปลี่ยนแปลงความละเอียดเชิงพื้นที่ spatial resolution ของข้อมูลต้นฉบับ เช่น ภาพ Landsat-9 ซึ่งมีความละเอียดเชิงพื้นที่ 30 เมตร

การปรับความคมชัดช่วยเพิ่มความแตกต่างเชิงสเปกตรัมระหว่างสิ่งของหรือสิ่งปลูฏสร้างบนพื้นผิวโลก ทำให้ลักษณะเชิงพื้นที่ spatial feature และขอบเขตของวัตถุนั้นชัดเจนมากขึ้น ส่งผลให้การแปลภาพด้วยสายตาและการวิเคราะห์การใช้ประโยชน์ที่ดิน  LU/LC มีความถูกต้องมากขึ้น และลดความคลาดเคลื่อนจากการตีความภาพ

กระบวนการนี้จึงเป็นขั้นตอนสำคัญในการประมวลผลภาพ image preprocessing เพื่อเพิ่มประสิทธิภาพในการจำแนกประเภท การตรวจจับการเปลี่ยนแปลง และการวิเคราะห์ข้อมูลจากภาพถ่ายดาวเทียมอย่างมีประสิทธิภาพ

**2.** อธิบายขั้นตอนสำคัญที่เกี่ยวข้องกับการปรับความคมชัดด้วนเทคนิค Pan-sharpening ตั้งแต่การรับข้อมูลไปจนถึงการสร้างภาพที่ปรับความคมชัด เทคนิค Pan-sharpening มี กระบวนการสุ่มตัวอย่างใหม่ช่วยจัดแนวภาพหลายสเปกตรัมและภาพแพนโครมาติกอย่างไร

**Answer** การปรับความคมชัดด้วยเทคนิค Pan-sharpening เป็นกระบวนการผสานข้อมูลจากภาพแพนโครมาติก (Panchromatic: PAN) ซึ่งมีความละเอียดของภาพเชิงพื้นที่สูง และภาพหลายสเปกตรัม Multispectral ซึ่งมีข้อมูลเชิงสเปกตรัมหรือสีหลายช่วงคลื่น เพื่อสร้างภาพใหม่ที่มีทั้งรายละเอียดเชิงพื้นที่สูงและคงคุณลักษณะเชิงสเปกตรัมของภาพต้นฉบับ เช่น ภาพจากดาวเทียม Landsat-9

(1)การรรับข้อมูล

เริ่มจากการรับข้อมูลภาพ PAN และ Multispectral ของดาวเทียม Landsat-9 ซึ่งเป็นข้อมูลที่ถ่ายในช่วงเวลาเดียวกัน หรือใกล้เคียงกัน และครอบคลุมพื้นที่เดียวกันทั้งหมด เพื่อหลีกเลี่ยงความคลาดเคลื่อนเชิงเวลา temporal error ที่อาจส่งผลต่อการผสานของภาพ

(2) การปรับแก้ข้อมูลเบื้องต้น Preprocessing

ก่อนทำ Pan-sharpening จำเป็นต้องเตรียมข้อมูลให้พร้อม โดยตรวจสอบและปรับแก้ดังนี้

ระบบพิกัดและตำแหน่งเชิงเรขาคณิต (Geometric correction) ให้ภาพ PAN และ MS อยู่ในระบบพิกัดเดียวกัน

ต่อจากนั้นจึงทำการปรับแก้เชิงรังสี (Radiometric correction) หรือแปลงค่า DN เป็น reflectance เพื่อให้ค่าความสว่างของภาพมีความสอดคล้องกัน
ขั้นตอนนี้ช่วยให้ภาพทั้งสองชุดสามารถนำมาผสานกันได้อย่างถูกต้อง

(3) การสุ่มตัวอย่างใหม่ Resampling

เนื่องจากภาพ PAN และ MS มีความละเอียดเชิงพื้นที่แตกต่างกัน (เช่น PAN = 15 เมตร และ MS = 30 เมตร) จึงต้องทำการสุ่มตัวอย่างใหม่ของภาพ MS ให้มีขนาดพิกเซลเท่ากับภาพ PAN โดยเลือกภาพ PAN เป็นภาพอ้างอิงผ่านกระบวนการใช้แบบ reference image

ซึ่งกระบวนการ Resampling เป็นการคำนวณค่าพิกเซลใหม่ของภาพ MS เพื่อให้ตำแหน่งพิกเซลของภาพหลายสเปกตรัมและภาพแพนโครมาติกตรงกันแบบ pixel-to-pixel ซึ่งช่วยจัดแนวภาพทั้งสองชุดในเชิงพื้นที่ หากขาดการดำเนินการขั้นตอนนี้ การผสานภาพจะเกิดความคลาดเคลื่อนของตำแหน่งและทำให้ภาพผลลัพธ์ผิดเพี้ยนได้

(4) การผสานภาพด้วยเทคนิค Pan-sharpening

เมื่อภาพ MS ถูก Resampling แล้ว จะนำมาผสานกับภาพ PAN โดยใช้เทคนิค Pan-sharpening ด้วยการ Brovey Transform หรือ IHS Transform หรือ PCA ซึ่งแต่ละวิธีจะนำข้อมูลเชิงพื้นที่จากภาพ PAN มารวมกับข้อมูลเชิงสเปกตรัมของภาพ MS เพื่อเพิ่มความคมชัดของภาพสี

(5) การสร้างและตรวจสอบภาพที่ปรับความคมชัด  
ผลลัพธ์จากกระบวนการ Pan-sharpening จะเป็นภาพสีที่มีความละเอียดเชิงพื้นที่สูงขึ้น โดยยังคงสีและข้อมูลเชิงสเปกตรัมจาก Landsat-9 ภาพดังกล่าวสามารถนำไปใช้ในการวิเคราะห์การใช้ประโยชน์ที่ดิน การตรวจจับการเปลี่ยนแปลง หรือการแปลภาพด้วยสายตาได้อย่างมีประสิทธิภาพและละเอียดมากขึ้น